# Codegen Trace: Vector Add Mask 构建过程

通过 monkey-patch 注入 trace，追踪 `Tensor.offsets()` 和 `_generate_offsets_and_mask()` 的调用链。


In [1]:
import sys
sys.path.insert(0, "/data/ninetoothed/src")
import matplotlib
matplotlib.use('Agg')

import inspect
import textwrap
import functools

from ninetoothed import Symbol, Tensor
from ninetoothed.generation import CodeGenerator
import ninetoothed


## 1. 真实源码

关键方法的原始代码，直接 `inspect.getsource()` 展示。


In [2]:
for name, obj in [
    ("Tensor.offsets()", Tensor.offsets),
    ("_generate_offsets_and_mask()", CodeGenerator._generate_offsets_and_mask),
    ("_generate_overall_offsets_and_mask()", CodeGenerator._generate_overall_offsets_and_mask),
    ("_generate_pointers_and_mask()", CodeGenerator._generate_pointers_and_mask),
]:
    print("=" * 70)
    print(f"{name}  -- {inspect.getfile(obj)}")
    print("=" * 70)
    print(textwrap.dedent(inspect.getsource(obj)))
    print()


Tensor.offsets()  -- /data/ninetoothed/src/ninetoothed/tensor.py
def offsets(self):
    indices = tuple(sum(indices) for indices in zip(*self._inputs))

    outputs = self._offsets(indices)

    for index, size in zip(indices, self.shape):
        index = Symbol(index)

        self.source._mask &= index < size
        self.source._mask &= index >= 0

    for output_, output in zip(self._outputs, outputs):
        output_.clear()
        output_.extend(output)


_generate_offsets_and_mask()  -- /data/ninetoothed/src/ninetoothed/generation.py
@staticmethod
def _generate_offsets_and_mask(tensor, indices):
    offsets = [Symbol(0) for _ in range(tensor.source.ndim)]

    tensor.source._mask = Symbol(True)

    curr = tensor
    start = 0

    while isinstance(curr, type(tensor)):
        stop = start + curr.ndim
        curr_indices = indices[start:stop]

        curr._inputs = [curr_indices]

        start = stop
        curr = curr.dtype

    for level in reversed(tensor._levels):
     

## 2. 注入 Trace（不改变行为）

wrapper 模式：调用原始方法前打印，返回结果后打印，不修改任何返回值。


In [3]:
# 保存原始方法
_orig = {
    "offsets": Tensor.offsets,
    "gen_offsets_mask": CodeGenerator._generate_offsets_and_mask,
    "gen_overall": CodeGenerator._generate_overall_offsets_and_mask,
    "gen_pointers": CodeGenerator._generate_pointers_and_mask,
}

# ---- Tensor.offsets trace ----
@functools.wraps(Tensor.offsets)
def _trace_offsets(self):
    indices = tuple(sum(indices) for indices in zip(*self._inputs))
    outputs = self._offsets(indices)
    print(f"  [offsets] shape={tuple(self.shape)}, ndim={self.ndim}")
    print(f"    _inputs={self._inputs}")
    print(f"    indices={indices}")
    print(f"    _offsets->{outputs}")
    for index, size in zip(indices, self.shape):
        print(f"    mask: {index} < {size}")
        print(f"    mask: {index} >= 0")
    for output_, output in zip(self._outputs, outputs):
        print(f"    _outputs[0]={output_} .clear() .extend({output})")
    return _orig["offsets"](self)

Tensor.offsets = _trace_offsets

# ---- _generate_offsets_and_mask trace ----
@functools.wraps(CodeGenerator._generate_offsets_and_mask)
def _trace_gen_offsets_mask(tensor, indices):
    print(f"\n  >>> _generate_offsets_and_mask")
    print(f"  tensor.shape={tuple(tensor.shape)}")
    print(f"  tensor.source.shape={tuple(tensor.source.shape)}")
    print(f"  tensor._levels={tensor._levels}")
    print(f"  indices={indices}")
    result = _orig["gen_offsets_mask"](tensor, indices)
    offsets, mask = result
    print(f"  return offsets={offsets}")
    print(f"  return mask={mask}")
    return result

CodeGenerator._generate_offsets_and_mask = staticmethod(_trace_gen_offsets_mask)

# ---- _generate_overall_offsets_and_mask trace ----
@functools.wraps(CodeGenerator._generate_overall_offsets_and_mask)
def _trace_gen_overall(tensor, indices):
    print(f"\n  >>> _generate_overall_offsets_and_mask")
    print(f"  tensor.shape={tuple(tensor.shape)}")
    print(f"  indices={indices}")
    result = _orig["gen_overall"](tensor, indices)
    overall_offsets, mask = result
    print(f"  overall_offsets={overall_offsets}")
    print(f"  return mask={mask}")
    return result

CodeGenerator._generate_overall_offsets_and_mask = staticmethod(_trace_gen_overall)

# ---- _generate_pointers_and_mask trace ----
@functools.wraps(CodeGenerator._generate_pointers_and_mask)
def _trace_gen_pointers(self, tensor, indices):
    print(f"\n>>> _generate_pointers_and_mask")
    print(f"tensor.shape={tuple(tensor.shape)}")
    if tensor is not tensor.source:
        pid = tuple(self._generate_pid_indices(tensor))
        inner = tuple(CodeGenerator._generate_innermost_indices(tensor))
        full = pid + indices + inner
        print(f"indices = {pid} + () + {inner}")
        print(f"       = {full}")
    result = _orig["gen_pointers"](self, tensor, indices)
    pointers, mask = result
    print(f"pointers={pointers}")
    return result

CodeGenerator._generate_pointers_and_mask = _trace_gen_pointers

print("Trace injected. Ready to generate.")


Trace injected. Ready to generate.


## 3. 触发代码生成

Triton 会对每个符号参数（lhs, rhs, output）分别调用 `_generate_pointers_and_mask`，所以 trace 会重复 3 次。


In [4]:
BLOCK_SIZE = Symbol("BLOCK_SIZE", meta=True)

@ninetoothed.jit
def add_kernel(
    lhs: Tensor(1).tile((BLOCK_SIZE,)),
    rhs: Tensor(1).tile((BLOCK_SIZE,)),
    output: Tensor(1).tile((BLOCK_SIZE,)),
):
    output = lhs + rhs

print("\nDone. Cached at:", add_kernel._source)



>>> _generate_pointers_and_mask
tensor.shape=((ninetoothed_ninetoothed_tensor_0_size_0 - (ninetoothed_meta_prefix_BLOCK_SIZE - 1) - 1 + ninetoothed_meta_prefix_BLOCK_SIZE - 1) // ninetoothed_meta_prefix_BLOCK_SIZE + 1,)
indices = (ninetoothed_tensor_0_index_0,) + () + (ninetoothed.language.arange(0, ninetoothed_meta_prefix_BLOCK_SIZE)[slice(None, None, None),],)
       = (ninetoothed_tensor_0_index_0, ninetoothed.language.arange(0, ninetoothed_meta_prefix_BLOCK_SIZE)[slice(None, None, None),])

  >>> _generate_overall_offsets_and_mask
  tensor.shape=((ninetoothed_ninetoothed_tensor_0_size_0 - (ninetoothed_meta_prefix_BLOCK_SIZE - 1) - 1 + ninetoothed_meta_prefix_BLOCK_SIZE - 1) // ninetoothed_meta_prefix_BLOCK_SIZE + 1,)
  indices=(ninetoothed_tensor_0_index_0, ninetoothed.language.arange(0, ninetoothed_meta_prefix_BLOCK_SIZE)[slice(None, None, None),])

  >>> _generate_offsets_and_mask
  tensor.shape=((ninetoothed_ninetoothed_tensor_0_size_0 - (ninetoothed_meta_prefix_BLOCK_SIZE - 1)

## 4. Trace 解读

从 trace 可以清晰看到 mask 构建流程：

1. `_generate_pointers_and_mask` 组装 indices：
   - `pid_indices` = `(ninetoothed_tensor_0_index_0,)` — 每个 CUDA block 的 ID
   - `innermost_indices` = `(arange(0, BLOCK_SIZE),)` — block 内的元素偏移
   - 最终 indices = `(pid, arange)`

2. `_generate_offsets_and_mask` 遍历 dtype 层级：
   - **Level 0 (outer)**: shape=num_blocks, `_inputs = [[pid]]`
   - **Level 1 (inner)**: shape=BLOCK_SIZE, `_inputs = [[arange]]`

3. 逆序遍历 `_levels`，调用 `offsets()`：
   - **outer.offsets()**: `_inputs=[[pid]]`, 追加 `pid < num_blocks` 和 `pid >= 0`
   - **inner.offsets()**: `_inputs=[[arange]]`, 追加 `arange < BLOCK_SIZE` 和 `arange >= 0`
   - **source.offsets()**: `_inputs=[[pid*BLOCK], [arange]]`, 追加 `pid*BLOCK+arange < N` 和 `pid*BLOCK+arange >= 0`

4. `_generate_overall_offsets_and_mask` 计算 `offsets[0] * stride_0` = `(pid*BLOCK + arange) * stride`

第 3 步中 source.offsets() 之所以有 `_inputs=[[pid*BLOCK], [arange]]`，是因为 outer.offsets() 和 inner.offsets() 通过 `_outputs[0]` 写回了 source 的 `_inputs[0]` 和 `_inputs[1]`。

最终 mask 包含 7 个条件，其中 `pid >= 0`、`arange < BLOCK`、`arange >= 0`、`pid*BLOCK+arange >= 0` 永远为真，`pid < num_blocks` 在 `N % BLOCK == 0` 时被覆盖。


## 5. 查看最终生成代码


In [5]:
# 恢复原始方法
Tensor.offsets = _orig["offsets"]
CodeGenerator._generate_offsets_and_mask = _orig["gen_offsets_mask"]
CodeGenerator._generate_overall_offsets_and_mask = _orig["gen_overall"]
CodeGenerator._generate_pointers_and_mask = _orig["gen_pointers"]

# 重新生成
del add_kernel
BLOCK_SIZE = Symbol("BLOCK_SIZE", meta=True)

@ninetoothed.jit
def add_kernel(
    lhs: Tensor(1).tile((BLOCK_SIZE,)),
    rhs: Tensor(1).tile((BLOCK_SIZE,)),
    output: Tensor(1).tile((BLOCK_SIZE,)),
):
    output = lhs + rhs

with open(add_kernel._source) as f:
    source = f.read()

# 提取 tl.load/tl.store 行（只显示前 200 字符）
for line in source.split('\n'):
    if 'tl.load' in line or 'tl.store' in line:
        print(line.strip()[:200])
        print("  ...")
        print()
